In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


In [22]:
import pandas as pd
df = pd.read_csv("demo.csv", sep='\t')

model_type="alphagenome"
ckpt_path="/home/wangyucheng/projects/Benchmark/Data_v2/EpisomalMPRA/K562/K562_complete_alphagenome/best_loss_epoch1_loss0.2687.pt"
    
BG_FREQ = {"EpisomalMPRA_K562": {"A": 0.27137775616174226, "C": 0.2270740421793547, "G": 0.22897209976197583, "T": 0.2725761018969272}}


LENGTH = 200          
BATCH_SIZE = 1280     
BACKGROUND_NUM = 30   
WINDOW_SIZE = 10  
FLANK_SIZE = 2   
TARGET_FDR = 0.05
MIN_PASSING_WINDOWS_FRAC = 0.1
MAX_PASSING_WINDOWS_FRAC = 0.4
WEAK_THRESHOLD_FOR_COUNTING_SIGN = 0.8

TAU = 0.75
OVERLAP_MIN = 4

## 1. Build the SeqTrust evaluator

All sequences are processed as one single collection (no grouping).

In [23]:
import sys
sys.path.insert(0, "/home/wangyucheng/public_repository/SeqTrust")

from seqtrust import SeqTrust

st = SeqTrust(df, sequence_col="sequence")
st.preview()

,sequence,functional_score
0,AAACAATCCTCAGAGGAAGCGCTATGGGGCGATCATACGTCCGAAAAGAATATTAGAAGATGAACTTCTCGTCACGCTTCAGCCGGTTTCCCATACGCTAGATAGTCGCGCGCGCCGCAGCGGAAACTTAAAGATGCCCGGAAGTTTTGTAATTGCGGCAAGAACGTAAGGCATGAGAGCTCAACATATTTTACCGTGAC,-1.826650
1,AAACAATCGTCAGAGGTAGCGCTATGGGGCGCTCATATCGAAGGTACAGCGACCATACTCTGACGCTGTACGGAAGTACCACGCACGCCGACTTCCGTCCGCTTTCGAGTCGCTTGGGTTCGTATTTACGTAGGCGCCCGGAAGTTGTGTCATTTGGGCAAGAACTTAAGCGTGCGACAGAAGTAGCGTCACCGACTTCC,0.099458
2,AAACGAACGTGTTGCTTACGCGCGGCGCACCGCTCAATAGCAATGGTGCGGATTGTCGTGCTGTCTCCCTACGTGTCATGGGCGCTTGATTTTTCAACACATCTACACTGCTCTCGTACGTGCGTCGACGTCATTTAAGAGGCGCCGACGCTCTATGAGGTACTCGCCAATCGGCATGTTGTACTTTGGGTTTACTTCCG,-0.187901
3,AAATTGCAGCTTAGTTGGGAATGGTTGCACTCCCGTCAGCTTCACGTGTTCGTATTGTGCGTTATACTCGGGCTTTACGGTCGCAGGAAGTGATGACGTGAGCTCAGGTGTTGACTTGGTGCGTCATTTCCGGAACCGGATGCGTCATTTACGGCGCAGGTACGTCATCGGAACCGCCATCGTGCTTCCGTGTGTCACTC,0.426452
4,AACAGAGAAATTGTCGCCTATGGCATTAAGCTCTGCACACTTCGCACGTCTGTCTAACTTATCTAACGTCAAGGGGGGGAAGGGTACACGCTTAGTCTTCGACTCCCAATCCTACTATCGTAAAGTGATTTCGGTAGGTCTCTTACGCATACAGCTGCCGTCCGAATGTAAGTCGCTACATGCCTTCGAAGGCGGCCAGT,-1.829194


## 2. Prediction scores (add_seq2func_prediction)

Load the fine-tuned checkpoint and predict the activity of each sequence into the `predicted_activity` column.

In [24]:
st.add_seq2func_prediction(
    model_type=model_type,
    output_col="predicted_activity",
    load_ckpt=True,
    ckpt_path=ckpt_path,
    length=LENGTH,
    batch_size=BATCH_SIZE,
)

[INFO] Using pretrained path:
/home/wangyucheng/projects/Benchmark/Tools/Predict_models/Seq2Func_pytorch/huggingface/gtca/alphagenome_pytorch/model_all_folds.safetensors
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/Benchmark/Data_v2/EpisomalMPRA/K562/K562_complete_alphagenome/best_loss_epoch1_loss0.2687.pt


100%|██████████| 1/1 [00:00<00:00, 12.24it/s]


## 3. Get embeddings (add_seq2func_embedding)

Extract trunk outputs of the model as sequence embeddings into the `alphagenome_embedding` column.

In [5]:
st.add_seq2func_embedding(
    model_type=model_type,
    output_col="alphagenome_embedding",
    load_ckpt=True,
    ckpt_path=ckpt_path,
    length=LENGTH,
    batch_size=BATCH_SIZE,
)

[INFO] Using pretrained path:
/home/wangyucheng/projects/Benchmark/Tools/Predict_models/Seq2Func_pytorch/huggingface/gtca/alphagenome_pytorch/model_all_folds.safetensors
Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/Benchmark/Data_v2/EpisomalMPRA/K562/K562_complete_alphagenome/best_loss_epoch1_loss0.2687.pt


100%|██████████| 1/1 [00:00<00:00, 16.33it/s]


## 4. Get attributions (add_seq2func_attribution)

Compute per-base attribution maps of shape (N, L, 4), stored in `st.results["attribution_alphagenome"]`.

In [6]:
st.add_seq2func_attribution(
    model_type=model_type,
    load_ckpt=True,
    ckpt_path=ckpt_path,
    length=LENGTH,
    batch_size=BATCH_SIZE,
    store_key="attribution_alphagenome",
    background_num=BACKGROUND_NUM,
)

Trunk output dim: 3072
Loaded checkpoint: /home/wangyucheng/projects/Benchmark/Data_v2/EpisomalMPRA/K562/K562_complete_alphagenome/best_loss_epoch1_loss0.2687.pt


Attribution: 100%|██████████| 1/1 [00:50<00:00, 50.99s/it]


In [7]:
st.results.keys()

dict_keys(['attribution_alphagenome'])

## 5. Scan motifs (add_motif)

Scan sequences with FIMO against the HOCOMOCO motif database, using the measured EpisomalMPRA_K562 background nucleotide frequencies (`bg_freq`).
Motif features are written to the `motif_features` column, and the positional scan table is stored in `st.results["motif_scan"]`.

In [8]:
st.add_motif(
    bg_freq=BG_FREQ["EpisomalMPRA_K562"],
    show_progress=True,
    output_type="binary",
    threshold=1e-4,
    qval_cutoff=0.05,
    pval_cutoff=1e-4,
)

Reading MEME file: /home/wangyucheng/public_repository/SeqTrust/seqtrust/tools/H14CORE_meme_format.meme
Successfully read 1595 motifs from HOCOMOCO
No duplicated motif names found.
Using user-specified background: A=0.2714, C=0.2271, G=0.2290, T=0.2726, GC=0.4560


Scanning motifs: 100%|██████████| 1595/1595 [00:34<00:00, 45.70it/s]


## 6. Get seqlets (add_seqlet)

Extract seqlets from the attribution maps with modisco-lite (purely attribution-based, no motif information involved).
No columns are added to `st.df` — the seqlet result dictionary is stored in `st.results["seqlets"]` and also saved as `PARM_seqlets.pkl`; metric columns are computed later by `add_metrics` (step 8).

In [ ]:
st.add_seqlet(
    attribution_key="attribution_alphagenome",
    window_size=WINDOW_SIZE,
    flank_size=FLANK_SIZE,
    target_fdr=TARGET_FDR,
    min_passing_windows_frac=MIN_PASSING_WINDOWS_FRAC,
    max_passing_windows_frac=MAX_PASSING_WINDOWS_FRAC,
    weak_threshold_for_counting_sign=WEAK_THRESHOLD_FOR_COUNTING_SIGN,
    task="Demo",
    save_path="demo_seqlets.pkl",
)

[Seqlets] threshold=0.163626, total=995, positive=970, negative=25, 336/392 sequences have >=1 seqlet
[Seqlets] Saved seqlets -> /home/wangyucheng/public_repository/SeqTrust/seqtrust/example/PARM_seqlets.pkl


In [10]:
st.results.keys()

dict_keys(['attribution_alphagenome', 'motif_scan', 'seqlets'])

## 7. Save the evaluator (optional)

Save the complete evaluator (including df, history, and the attribution / motif_scan / seqlets stored in results) so it can be reloaded later to skip all computation.

In [ ]:
# st.save("/home/wangyucheng/public_repository/SeqTrust/seqtrust/example/demo_seqtrust.pkl")

# Reload later to skip all computation:
# st = SeqTrust.load("demo_seqtrust.pkl")

## 8. Compute metrics (add_metrics)

Compute the 46 seqlet-based plausibility metrics of each sequence and add them as columns to `st.df`: count / coverage / span / pairwise uniqueness / diversity, matched seqlet-motif features, single-base negative attribution, self-redundancy, combinatorial pairs and adjacent-heterotypic seqlets, each over the positive, motif-augmented and valid (entropy >= `tau`) seqlet sets.

Seqlets are read directly from `st.results["seqlets"]` (step 6); alternatively pass `seqlets_path` to load the seqlet pickle saved in step 6. All thresholds default to `seqtrust.tools.metric.DEFAULT_PARAMS` and can be overridden per call.

In [ ]:
st.add_metrics(
    attribution_key="attribution_alphagenome",
    task="Demo",
    tau=TAU,
    overlap_min=OVERLAP_MIN,
)

# Seqlets can also be loaded from the pkl saved in step 6 instead of st.results:
# st.add_metrics(
#     seqlets_path="demo_seqlets.pkl",
#     attribution_key="attribution_alphagenome",
#     task="Demo",
# )
# Extra: collect_pairs=True also stores the long-form combinatorial pair tables in
# st.results["combinatorial_motif_pairs"] and st.results["combinatorial_all_pairs"].

[Metrics] seqlets from results['seqlets']: kept 856 positive seqlets (overlap suppression removed 114)
PARM: pos seqlets  n mean=2.18, zero=57/392 | coverage mean=30.4 bp | span mean=58.7 bp
PARM: augmented    +1.17 motif-augmented seqlets -> n mean=3.36 | valid (raw) n mean=1.78 | valid (aug) n mean=2.71
PARM: diversity    raw=0.415 | aug=0.432 | valid=0.377 | valid-aug=0.419
PARM: matched-motif seqlets  raw=1.81 | aug=2.74 | valid-aug=2.35 | unique features raw=1.24
PARM: single_base=0.45 | sr=0.03 | motif-pairs=0.18 | all-pairs=0.30 | adjacent-heterotypic=0.95 (raw-set means)


In [14]:
st.df.columns

Index(['sequence', 'functional_score', 'PARM_predicted_functional_score',
       'predicted_activity', 'alphagenome_embedding', 'motif_features',
       'n_pos_seqlets', 'pos_seqlets_coverage', 'pos_seqlets_span',
       'n_motif_seqlets_added', 'n_pos_seqlets_aug',
       'pos_seqlets_coverage_aug', 'pos_seqlets_span_aug',
       'n_valid_pos_seqlets', 'valid_pos_seqlets_coverage',
       'valid_pos_seqlets_span', 'n_valid_pos_seqlets_aug',
       'valid_pos_seqlets_coverage_aug', 'valid_pos_seqlets_span_aug',
       'pos_seqlets_pair_uniq', 'pos_seqlets_pair_uniq_aug',
       'valid_pos_seqlets_pair_uniq', 'valid_pos_seqlets_pair_uniq_aug',
       'pos_seqlets_nonred_partner_num', 'pos_seqlets_nonred_partner_num_aug',
       'valid_pos_seqlets_nonred_partner_num',
       'valid_pos_seqlets_nonred_partner_num_aug',
       'n_pos_seqlet_with_matched_motif',
       'n_pos_seqlet_with_matched_motif_aug',
       'n_valid_pos_seqlet_with_matched_motif_aug',
       'n_unique_matched_feature